In [9]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
import numpy as np

# 🥊 Bootstrap Smackdown!
### Estimating Regression Slopes with Bootstrap Confidence Intervals

**The Challenge:** Your team will use bootstrap sampling to estimate a regression slope
and its 95% confidence interval. Then you'll answer the key question:

> *Is our slope statistically real — or could it just be random noise?*

---
## Competition Rules

| Step | Task | Points |
|------|------|--------|
| 1 | Scatter plot of your team's data | 1 |
| 2 | Observed slope (no bootstrap yet) | 1 |
| 3 | Working bootstrap loop (1000 samples) | 3 |
| 4 | Correct 95% CI on the slope | 2 |
| 5 | CI for prediction at the mean x | 2 |
| 6 | Final Smackdown Figure | 2 |
| 7 | Correct interpretation: *"Is the slope real?"* | 2 |

**⚡ Fastest correct answer wins the Smackdown!**

---

![Philly Smackdown](Philly_smackdown.png)

## Regression Tools
These are the same helper functions from lecture — run this cell first.

In [10]:
def standard_units(arr):
    """Convert an array to standard units (z-scores)."""
    return (arr - np.mean(arr)) / np.std(arr)

def correlation(t, x_col, y_col):
    """Pearson correlation coefficient for two columns of a Table."""
    return np.mean(standard_units(t.column(x_col)) * standard_units(t.column(y_col)))

def slope(t, x_col, y_col):
    """Regression slope for predicting y_col from x_col."""
    r = correlation(t, x_col, y_col)
    return r * np.std(t.column(y_col)) / np.std(t.column(x_col))

def intercept(t, x_col, y_col):
    """Regression intercept for predicting y_col from x_col."""
    return np.mean(t.column(y_col)) - slope(t, x_col, y_col) * np.mean(t.column(x_col))

## 🗂️ Team Datasets

Each team runs **exactly one** of the five dataset cells below.
After your cell runs, all remaining notebook cells will work automatically —
they use the shared variable names `my_table`, `x_col`, `y_col`.

| Team | Dataset | X variable | Y variable | Rows |
|------|---------|-----------|-----------|------|
| 🌋 Faithful | Old Faithful geyser | Eruption duration (min) | Wait time (min) | 272 |
| 📐 Galton | Parent–child heights | Mid-parent height (in) | Child height (in) | 898 |
| 👶 Baby | Newborn births | Gestational days | Birth weight (oz) | 1174 |
| 🎓 Census | ACS 2023 by state | % Bachelor's degree | Median income ($k) | 50 |
| 🏎️ Roads | Stopping distances | Speed (mph) | Stopping distance (ft) | 50 |

---

### 🌋 Team Faithful — Run this cell if you are Team Faithful

In [ ]:
# === TEAM FAITHFUL 🌋 ===
# Old Faithful geyser eruptions in Yellowstone National Park
my_table = Table.read_table('../Lab08/faithful-new.csv')
# Backup URL if the local file isn't available:
# my_table = Table.read_table('https://raw.githubusercontent.com/data-8/textbook/master/assets/data/faithful.csv')
x_col   = 'duration'
y_col   = 'wait'
x_label = 'Eruption Duration (minutes)'
y_label = 'Wait Time (minutes)'
team_name = 'Team Faithful 🌋'
print(f"{team_name} loaded — {my_table.num_rows} rows")
my_table.show(3)

### 📐 Team Galton — Run this cell if you are Team Galton

In [ ]:
# === TEAM GALTON 📐 ===
# Francis Galton's 1886 parent-child height study (the origin of "regression"!)
my_table = Table.read_table('https://raw.githubusercontent.com/data-8/textbook/master/assets/data/heights.csv')
x_col   = 'MidParent'
y_col   = 'Child'
x_label = 'Mid-Parent Height (inches)'
y_label = 'Child Height (inches)'
team_name = 'Team Galton 📐'
print(f"{team_name} loaded — {my_table.num_rows} rows")
my_table.show(3)

### 👶 Team Baby — Run this cell if you are Team Baby

In [ ]:
# === TEAM BABY 👶 ===
# Birth records from a Northern California hospital, 1960s
my_table = Table.read_table('https://raw.githubusercontent.com/data-8/textbook/master/assets/data/baby.csv')
x_col   = 'Gestational Days'
y_col   = 'Birth Weight'
x_label = 'Gestational Days'
y_label = 'Birth Weight (ounces)'
team_name = 'Team Baby 👶'
print(f"{team_name} loaded — {my_table.num_rows} rows")
my_table.show(3)

### 🎓 Team Census — Run this cell if you are Team Census

In [11]:
# === TEAM CENSUS 🎓 ===
# U.S. Census Bureau — American Community Survey (ACS) 2023
# State-level: Median Household Income vs. % Adults with a Bachelor's Degree or Higher

states = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
    'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
    'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
    'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
]
# % adults 25+ with Bachelor's degree or higher (ACS 2023)
pct_bachelors = [
    25.8, 30.9, 31.7, 24.0, 35.7, 43.5, 40.8, 35.6, 31.6, 32.5,
    35.2, 29.4, 37.0, 26.9, 31.4, 33.9, 25.5, 25.2, 36.3, 43.2,
    47.2, 31.5, 38.7, 22.3, 31.7, 33.3, 34.5, 27.2, 40.4, 42.6,
    27.2, 38.7, 33.6, 33.3, 30.1, 25.3, 36.6, 34.8, 37.4, 30.5,
    30.0, 28.0, 32.6, 36.6, 39.3, 42.8, 38.9, 21.1, 32.8, 27.8
]
# Median household income in $1,000s (ACS 2023)
median_income_k = [
    57.6, 84.5, 72.6, 55.4, 91.6, 87.6, 90.2, 79.3, 67.9, 71.2,
    87.5, 66.1, 78.2, 63.2, 72.4, 68.6, 59.0, 57.8, 73.3, 98.5,
    98.1, 68.7, 83.5, 50.0, 66.5, 63.2, 73.0, 68.5, 90.3, 99.5,
    55.7, 81.4, 66.5, 74.0, 65.7, 57.9, 75.8, 73.9, 79.9, 63.6,
    65.6, 62.2, 72.5, 82.2, 74.0, 87.7, 90.3, 50.9, 72.7, 70.6
]

my_table = Table().with_columns(
    'pct_bachelors', pct_bachelors,
    'median_income_k', median_income_k,
    'state', states
)
x_col     = 'pct_bachelors'
y_col     = 'median_income_k'
x_label   = '% Adults with Bachelor\'s Degree or Higher'
y_label   = 'Median Household Income ($1,000s)'
team_name = 'Team Census 🎓'
print(f"{team_name} loaded — {my_table.num_rows} rows (one per U.S. state)")
my_table.show(5)

Team Census 🎓 loaded — 50 rows (one per U.S. state)


pct_bachelors,median_income_k,state
25.8,57.6,AL
30.9,84.5,AK
31.7,72.6,AZ
24,55.4,AR
35.7,91.6,CA


### 🏎️ Team Roads — Run this cell if you are Team Roads

In [ ]:
# === TEAM ROADS 🏎️ ===
# Speed and stopping distance of 1920s automobiles (classic R dataset)
speed = [4,4,7,7,8,9,10,10,10,11,11,12,12,12,12,13,13,13,13,14,
         14,14,14,15,15,15,16,16,17,17,17,18,18,18,18,19,19,19,
         20,20,20,20,20,22,23,24,24,24,24,25]
dist  = [2,10,4,22,16,10,18,26,34,17,28,14,20,24,28,26,34,34,46,
         26,36,60,80,20,26,54,32,40,32,40,50,42,56,76,84,36,46,
         68,32,48,52,56,64,66,54,70,92,93,120,85]
my_table = Table().with_columns('speed', speed, 'distance', dist)
x_col   = 'speed'
y_col   = 'distance'
x_label = 'Speed (mph)'
y_label = 'Stopping Distance (feet)'
team_name = 'Team Roads 🏎️'
print(f"{team_name} loaded — {my_table.num_rows} rows")
my_table.show(3)

---
## Step 1 — Explore Your Data 📊
Make a scatter plot and get a feel for the relationship.

In [12]:
# TODO: Create a scatter plot of your data
# Hint: use my_table.scatter(x_col, y_col)
...

# Print a quick summary
print(f"X mean: {np.mean(my_table.column(x_col)):.3f}")
print(f"Y mean: {np.mean(my_table.column(y_col)):.3f}")
print(f"Number of rows: {my_table.num_rows}")

X mean: 33.258
Y mean: 73.338
Number of rows: 50


---
## Step 2 — Compute the Observed Slope 📐
Before bootstrapping, calculate the slope and intercept on the **full** dataset.
This is your best single estimate.

In [ ]:
# TODO: Compute the observed slope and intercept using the full table
obs_slope     = ...
obs_intercept = ...

print(f"Observed slope:     {obs_slope:.4f}")
print(f"Observed intercept: {obs_intercept:.4f}")
print(f"\nRegression line: {y_col} = {obs_slope:.3f} × {x_col} + {obs_intercept:.3f}")

---
## Step 3 — Bootstrap! 🥾
Sample your table **with replacement** 1000 times.
Each bootstrap sample gives you one estimate of the slope and intercept.

> **Key idea:** `.sample()` with no argument samples *n* rows with replacement,
> where *n* = the number of rows in your table. Each call gives a slightly
> different sample — and therefore a slightly different slope.

In [ ]:
N_bootstrap = 1000
sample_slopes     = []   # will hold 1000 bootstrap slopes
sample_intercepts = []   # will hold 1000 bootstrap intercepts

for i in np.arange(N_bootstrap):
    # TODO: Sample your table with replacement
    boot_sample = ...
    
    # TODO: Compute the slope for this bootstrap sample
    boot_slope = ...
    
    # TODO: Compute the intercept for this bootstrap sample
    boot_intercept = ...
    
    # Store both values
    sample_slopes.append(boot_slope)
    sample_intercepts.append(boot_intercept)

print(f"✅ Generated {len(sample_slopes)} bootstrap slopes!")
print(f"Bootstrap mean slope: {np.mean(sample_slopes):.4f}  (compare to observed: {obs_slope:.4f})")

---
## Step 4 — 95% Confidence Interval on the Slope 📏
Use the 2.5th and 97.5th percentiles of your bootstrap slopes as the
lower and upper bounds of a 95% confidence interval.

**Hypothesis test interpretation:**
- If the CI *excludes* 0 → the slope is statistically significant (correlation is real!)
- If the CI *includes* 0 → we can't rule out that the true slope is zero

In [ ]:
# TODO: Compute the left (2.5th percentile) and right (97.5th percentile) bounds
# Hint: use the datascience percentile() function
slope_CI_left  = ...
slope_CI_right = ...

CI_half_width = (slope_CI_right - slope_CI_left) / 2

print(f"95% CI on slope: [{slope_CI_left:.4f},  {slope_CI_right:.4f}]")
print(f"Reported as:  {obs_slope:.3f} ± {CI_half_width:.3f}")

# Does the CI exclude zero?
if slope_CI_left > 0 or slope_CI_right < 0:
    print(f"\n✅ CI excludes 0 — the slope is statistically significant!")
    print(f"   The variables ARE correlated.")
else:
    print(f"\n❌ CI includes 0 — the slope is NOT statistically significant.")
    print(f"   We cannot reject the null hypothesis of no correlation.")

---
## Step 5 — Confidence Interval for Prediction at the Mean 🎯
What is our best prediction of *y* when *x* equals its mean value?
And how uncertain are we?

For each bootstrap sample we already have a slope *and* an intercept,
so we can compute the predicted *y* at x̄ for all 1000 samples.

In [ ]:
# Compute the mean of x in the full dataset
x_mean = np.mean(my_table.column(x_col))
print(f"Mean of {x_col}: {x_mean:.3f}")

# For each bootstrap sample, predict y at x_mean
# predicted = slope * x_mean + intercept
predicted_at_mean = np.array(sample_slopes) * x_mean + np.array(sample_intercepts)

# TODO: Compute the 95% CI on predicted_at_mean
pred_CI_left  = ...
pred_CI_right = ...

# The best point estimate (using the full-data slope and intercept)
point_estimate = obs_slope * x_mean + obs_intercept

print(f"\nPredicted {y_col} at mean {x_col}:")
print(f"  Point estimate: {point_estimate:.3f}")
print(f"  95% CI:         [{pred_CI_left:.3f},  {pred_CI_right:.3f}]")

---
## Step 6 — The Smackdown Figure 🏆
Create a two-panel figure showing:
- **Left panel:** Scatter plot with your regression line
- **Right panel:** Bootstrap slope distribution with 95% CI

In [ ]:
team_label = team_name.encode('ascii', 'ignore').decode().strip()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Bootstrap Smackdown — {team_label}', fontsize=14, fontweight='bold')

# ── Left panel: Scatter + regression line ─────────────────────────────────────
ax1 = axes[0]
xs = my_table.column(x_col)
ys = my_table.column(y_col)

ax1.scatter(xs, ys, alpha=0.5, color='steelblue', s=20, label='Data')

# Draw regression line across the range of x
x_range = np.linspace(np.min(xs), np.max(xs), 200)
y_line  = obs_slope * x_range + obs_intercept
ax1.plot(x_range, y_line, color='crimson', lw=2,
         label=f'Slope = {obs_slope:.3f}')

# Mark the mean-x prediction point
ax1.scatter(x_mean, point_estimate, s=120, zorder=5,
            color='gold', edgecolors='black', linewidths=1.5,
            label=f'Predicted at mean x
= {point_estimate:.2f} [{pred_CI_left:.2f}, {pred_CI_right:.2f}]')

ax1.set_xlabel(x_label)
ax1.set_ylabel(y_label)
ax1.set_title('Data & Regression Line')
ax1.legend(fontsize=8)

# ── Right panel: Bootstrap slope histogram + CI ──────────────────────────────
ax2 = axes[1]
ax2.hist(sample_slopes, bins=30, color='steelblue', alpha=0.7, edgecolor='white')

# Mark the observed slope
ax2.axvline(obs_slope, color='crimson', lw=2, linestyle='--',
            label=f'Observed slope = {obs_slope:.3f}')

# Mark the 95% CI
ax2.plot([slope_CI_left, slope_CI_right], [0, 0],
         color='gold', lw=8, solid_capstyle='round',
         label=f'95% CI: [{slope_CI_left:.3f}, {slope_CI_right:.3f}]')
ax2.scatter(obs_slope, 0, s=150, color='crimson', zorder=5, edgecolors='black')

ax2.set_xlabel(f'Bootstrap Slope ({y_label} / {x_label})')
ax2.set_ylabel('Count')
ax2.set_title('Bootstrap Distribution of Slopes')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'smackdown_{team_name.split()[1]}.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved!")

---
## 📣 Report Your Results!

Fill in this template when presenting to the class:

> **[Team Name]** analyzed the relationship between **[X variable]** and **[Y variable]**.
>
> Our observed slope was **___**, meaning that for each one-unit increase in [X],
> [Y] changes by approximately ___ units.
>
> After 1000 bootstrap samples, our 95% confidence interval on the slope was
> **[ ___ ,  ___ ]**.
>
> Because this interval **does / does not** include zero, we **can / cannot** conclude
> that the variables are correlated.
>
> At the mean value of [X] = ___, we predict [Y] = ___ with a 95% CI of [ ___ ,  ___ ].

---
### 🤔 Discussion Questions

1. Interpret your result in terms of what the value of the slope means in parctical terms, give an example.
2. Which team had the **narrowest** confidence interval? Why might that be?
3. Which team had the **steepest** slope? What does that mean in context?
4. If you increased `N_bootstrap` from 1000 to 10000, what would change?
5. Why do we sample *with* replacement? What goes wrong if we sample without replacement?
